In [2]:
# ============================================
# RETAINIQ - EXPLORATORY DATA ANALYSIS
# ============================================

import matplotlib

# Use a stable non-interactive backend
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from IPython.display import display, Image

# Matplotlib configuration
matplotlib.rcParams["font.family"] = "DejaVu Sans"
matplotlib.rcParams["font.size"] = 10
matplotlib.rcParams["text.usetex"] = False
matplotlib.rcParams["mathtext.fontset"] = "dejavusans"

# Seaborn configuration
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")
print("Matplotlib version:", matplotlib.__version__)
print("Seaborn version:", sns.__version__)

Libraries imported successfully.
Matplotlib version: 3.11.0
Seaborn version: 0.13.2


In [3]:
# Load the Telco Customer Churn dataset

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# ============================================
# DATA CLEANING
# ============================================

# Convert TotalCharges from string to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Remove rows where TotalCharges could not be converted
df.dropna(subset=["TotalCharges"], inplace=True)

# Remove duplicate rows
df.drop_duplicates(inplace=True)

# Customer ID is an identifier, not a useful predictive feature
df.drop(columns=["customerID"], inplace=True)

# Convert target variable:
# Yes = 1
# No  = 0
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

print("Data cleaning completed.")
print("Final shape:", df.shape)

Data cleaning completed.
Final shape: (7032, 20)


In [5]:
# ============================================
# BASIC DATASET INSPECTION
# ============================================

print("Dataset Shape:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nStatistical Summary:")
display(df.describe())

Dataset Shape:
(7032, 20)

Data Types:
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

Missing Values:
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod     

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn
count,7032.000000,7032.000000,7032.000000,7032.000000,7032.000000
mean,0.162400,32.421786,64.798208,2283.300441,0.265785
std,0.368844,24.545260,30.085974,2266.771362,0.441782
min,0.000000,1.000000,18.250000,18.800000,0.000000
25%,0.000000,9.000000,35.587500,401.450000,0.000000
50%,0.000000,29.000000,70.350000,1397.475000,0.000000
75%,0.000000,55.000000,89.862500,3794.737500,1.000000
max,1.000000,72.000000,118.750000,8684.800000,1.000000


In [7]:
# ============================================
# 1. CHURN DISTRIBUTION
# ============================================

churn_counts = df["Churn"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))

ax.bar([0, 1], churn_counts.values)

# Remove all text rendering
ax.set_xticks([])
ax.set_yticks([])

plt.savefig("churn_distribution.png", dpi=100)
plt.close()

print("Churn distribution chart created successfully.")

Churn distribution chart created successfully.


In [8]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.family"] = "DejaVu Sans"
matplotlib.rcParams["font.size"] = 10

print("Matplotlib font configuration fixed.")

Matplotlib font configuration fixed.


In [9]:
# ============================================
# 2. CHURN BY CONTRACT TYPE
# ============================================

contract_churn = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
) * 100

contract_churn.columns = ["No Churn (%)", "Churn (%)"]

display(contract_churn.round(2))

,No Churn (%),Churn (%)
Contract,,
Month-to-month,57.29,42.71
One year,88.72,11.28
Two year,97.15,2.85


In [11]:
# ============================================
# 2. CHURN BY CONTRACT TYPE - VISUALIZATION
# ============================================

contract_counts = pd.crosstab(
    df["Contract"],
    df["Churn"]
)

# Create chart
fig, ax = plt.subplots(figsize=(8, 5))

contract_counts.plot(
    kind="bar",
    ax=ax,
    legend=False
)

# Remove all text from the figure
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

# Save without displaying
fig.savefig(
    "churn_by_contract.png",
    dpi=100,
    bbox_inches=None
)

plt.close(fig)

print("Churn by contract chart saved successfully.")

Churn by contract chart saved successfully.


In [12]:
# ============================================
# 3. TENURE VS CHURN
# ============================================

tenure_analysis = df.groupby("Churn")["tenure"].agg(
    ["count", "mean", "median", "min", "max"]
)

tenure_analysis.index = ["No Churn", "Churn"]

display(tenure_analysis.round(2))

,count,mean,median,min,max
No Churn,5163,37.65,38.0,1,72
Churn,1869,17.98,10.0,1,72


In [13]:
# ============================================
# 4. MONTHLY CHARGES VS CHURN
# ============================================

monthly_charges_analysis = df.groupby("Churn")["MonthlyCharges"].agg(
    ["count", "mean", "median", "min", "max"]
)

monthly_charges_analysis.index = ["No Churn", "Churn"]

display(monthly_charges_analysis.round(2))

,count,mean,median,min,max
No Churn,5163,61.31,64.45,18.25,118.75
Churn,1869,74.44,79.65,18.85,118.35


In [14]:
# ============================================
# 5. INTERNET SERVICE VS CHURN
# ============================================

internet_churn = pd.crosstab(
    df["InternetService"],
    df["Churn"],
    normalize="index"
) * 100

internet_churn.columns = ["No Churn (%)", "Churn (%)"]

display(internet_churn.round(2))

,No Churn (%),Churn (%)
InternetService,,
DSL,81.00,19.00
Fiber optic,58.11,41.89
No,92.57,7.43


In [15]:
# ============================================
# 6. NUMERICAL FEATURE CORRELATION
# ============================================

numeric_df = df.select_dtypes(include=np.number)

correlation = numeric_df.corr()

display(correlation.round(2))

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn
SeniorCitizen,1.00,0.02,0.22,0.10,0.15
tenure,0.02,1.00,0.25,0.83,-0.35
MonthlyCharges,0.22,0.25,1.00,0.65,0.19
TotalCharges,0.10,0.83,0.65,1.00,-0.20
Churn,0.15,-0.35,0.19,-0.20,1.00
